# CME Futures: Holdout Predictions

**Chapter 20 - Out-of-sample evaluation**

Every number in this case study so far was measured on the validation periods, and every
choice was made by looking at them: which model family, which configuration, how the
positions are sized, whether a risk overlay earns its place. A result selected that way
cannot also be evidence that the selection was sound - the ranking and the evidence would
be the same measurement.

The holdout is the window nothing has been selected on: 2024-01-01 to 2025-12-31. This
notebook fits the selected configuration on the history available before that window
opens and writes its predictions over it.
[`18_holdout_backtest`](18_holdout_backtest.ipynb) turns those predictions into a return
series with the sizing this case study settled on, and
[`19_strategy_analysis`](19_strategy_analysis.ipynb) reads both back.

**What this notebook is careful about**

A holdout prediction is not the validation model scored on a later window. Those two
differ in what the model saw while fitting, and only the first is out of sample. Section
3 fits again, and the new training identity is what makes the refit visible rather than
asserted - a run that came back carrying the validation training hash would mean no refit
happened.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：保留樣本預測

**第 20 章——樣本外評估**

截至目前，本案例研究中的每個數字都在驗證期間衡量，每項選擇也都是根據驗證期間做出的：模型家族、設定、部位規模，以及風險覆蓋層是否值得採用。以這種方式選出的結果，不能同時當成選擇正確的證據，因為排序依據與證據會是同一項衡量結果。

保留樣本是未曾用來做選擇的期間：2024-01-01 至 2025-12-31。本 Notebook 會用該期間開始前可用的歷史資料擬合選定設定，並產生該期間的預測。
[`18_holdout_backtest`](18_holdout_backtest.ipynb) 會依本案例研究最終採用的部位規模決定方式，將這些預測轉成報酬率序列；
[`19_strategy_analysis`](19_strategy_analysis.ipynb) 則會回頭解讀兩者。

**本 Notebook 特別留意的事項**

保留樣本預測，不等於把驗證期模型拿到較晚的期間評分。兩者在模型擬合時看過的資料不同；只有前者才是樣本外。第 3 節會重新擬合，而新的訓練識別資訊能證明重新擬合確實發生，而非只有口頭聲明。若執行結果仍帶有驗證期的訓練雜湊值，就表示沒有重新擬合。
<!-- ML4T_ZH_END c0000-p000 -->

There is a second thing this case study had to fix before the refit could mean anything.
`04_model_based_features` fits ARIMA and a hidden Markov model per period, so a period
the file never wrote carries no forecast and no regime probability. The holdout period
was being appended to the fold spec without those features being generated for it, which
left a model reading two of nine features over the window it is judged on. The artifact
now carries a holdout vintage: coefficients estimated at the last session before the
window opens, rolled forward across it frozen.

**Prerequisites:** [`16_costs`](16_costs.ipynb), which fixes the configuration the
holdout runs.

**Scope:** one training run and one prediction set. No backtest, no selection, no
comparison - those are 18 and 19.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
在重新擬合能有意義之前，本案例研究還必須修正第二件事。`04_model_based_features` 會逐期間擬合 ARIMA 與隱藏馬可夫模型，因此，若檔案從未寫入某個期間，該期間就沒有預測值或市場狀態機率。原本的做法是把保留樣本期間附加到折規格中，卻未為該期間產生這些特徵，使模型在受評估期間只能讀到九項特徵中的兩項。現在的產出檔包含保留樣本版本：在期間開始前的最後一個交易時段估計係數，並固定係數，將預測向前推進整個期間。

**先決條件：**[`16_costs`](16_costs.ipynb)，其會固定保留樣本要使用的設定。

**範圍：**一次訓練執行與一組預測。沒有回測、選擇或比較；這些工作屬於第 18 與第 19 份 Notebook。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""CME Futures: Holdout Predictions."""

import sqlite3

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.strategy_analysis import (
    resolve_solvent_carrier,
    training_run_fitted_for_the_holdout,
)
from case_studies.utils.warning_policy import apply_notebook_warning_policy
from utils.paths import get_case_study_dir

apply_notebook_warning_policy()

In [2]:
CASE_STUDY_ID = "cme_futures"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
# Whether a holdout generation for a DIFFERENT configuration may be superseded by this run.
# Off by default: see section 3.
#
# The flag exists because the holdout is not a one-shot resource, which is a ruling and not
# an oversight. What the rule against consulting the holdout forbids is SELECTING on it: the
# configuration evaluated here is chosen by validation backtest Sharpe, and no holdout number
# feeds back into that choice. It says nothing about how many times the evaluation may be
# computed, and a wrong result is deleted and re-run rather than left standing because it was
# observed. Reading the rule as a physical constraint is what produced a lock layer around
# this window, and it is being removed. The guard here is against something narrower and real:
# two generations readable at once, so nobody downstream has to choose between them and nobody
# can quote whichever number they prefer.

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)


def _registered_holdout_generations(case_dir):
    """Every holdout prediction set in the registry, and whether its model was refitted.

    `refitted` is read from the training run's own CV rather than from the prediction
    set's split: the split says where the predictions land, and a model fitted on the
    validation folds can publish predictions over the holdout window. That is the defect
    the generation below replaces, and it is why this is the same predicate the canonical
    lineage resolver applies.
    """
    with sqlite3.connect(str(case_dir / "run_log" / "registry.db")) as conn:
        rows = conn.execute(
            """
            SELECT p.prediction_hash, p.training_hash, p.checkpoint_kind, p.checkpoint_value,
                   t.config_name, t.spec_json
            FROM prediction_sets p
            JOIN training_runs t ON t.training_hash = p.training_hash
            WHERE p.split = 'holdout'
            ORDER BY p.prediction_hash
            """
        ).fetchall()
    return [
        {
            "prediction_hash": prediction_hash,
            "training_hash": training_hash,
            # The checkpoint is part of the configuration, not a detail of it: one training
            # run publishes one prediction set per declared checkpoint, and moving the
            # selection from one checkpoint to another is a different configuration
            # evaluated on the same window. Identity on the training hash alone would see
            # that as the same generation and let both stand.
            "checkpoint": (checkpoint_kind, checkpoint_value),
            "config_name": config_name,
            "refitted": training_run_fitted_for_the_holdout(training_spec_json),
        }
        for (
            prediction_hash,
            training_hash,
            checkpoint_kind,
            checkpoint_value,
            config_name,
            training_spec_json,
        ) in rows
    ]

## 1. Which configuration the holdout runs

The holdout runs the configuration the case study reports, resolved through the same
`resolve_solvent_carrier` [`16_costs`](16_costs.ipynb) uses. Resolving it again here
rather than passing it along is deliberate: the two notebooks must agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is
rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the validation
rank-1, and it was fixed before this notebook ran.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留樣本要使用哪項設定

保留樣本使用本案例研究報告的設定，並透過與 `resolve_solvent_carrier` [`16_costs`](16_costs.ipynb) 相同的方式解析。此處刻意重新解析，而非直接傳遞設定：兩份 Notebook 必須依建構方式就能一致；若只在其中一份記錄雜湊值，另一份再讀取，則僅在掃描結果尚未重建時才會一致。

保留樣本的任何資訊都不參與這項選擇。選定設定是驗證期排名第一者，且在本 Notebook 執行前就已固定。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

Selected configuration: 5075cacaecb2  stage=risk_overlay  family=latent_factors  config=sdf  label=fwd_ret_21d
  validation Sharpe 1.100, max drawdown -0.203
  fitted by training run 7e0f27986e67


The checkpoint is part of the configuration, and it is read from the selected configuration's own
prediction set rather than assumed. Every family this case study fits publishes at the
end of training, so the value here is `final` and carries no iteration - but reading it
is what keeps that true rather than asserted: a family that later publishes on a
schedule would name one of its checkpoints here, and refitting without it would produce
the model at the end of training instead of the one that was ranked.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是設定的一部分，應從選定設定自身的預測集讀取，而非直接假定。本案例研究擬合的每個模型家族都會在訓練結束時發布，因此此處的值為 `final`，不帶迭代次數；但唯有實際讀取，才能確保這是經驗證的事實，而非單純聲明。若某個模型家族往後改為定期發布檢查點，這裡就會指定其中一個；若重新擬合時未指定，產生的會是訓練結束時的模型，而非當初參與排名的模型。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

Checkpoint: epoch=768


## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from the case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the
three cannot disagree.

The training interval is everything available before that window, bounded above by a
label buffer. The buffer is what stops the last training label's outcome from resolving
inside the holdout: this case study dates each row by the session the decision is taken
on and the label looks forward from there, so a row dated inside a horizon's reach of
2024-01-01 has its outcome realised in the holdout. The buffer is therefore a horizon's
worth of sessions - 5 for `fwd_ret_5d`, 21 for `fwd_ret_21d` - and the derivation seals
on the widest of the declared labels rather than on the one being fitted, so no label
can reach past its own boundary. A zero gap would be a leak, not a conservative choice,
so the derivation refuses to default it.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a
fold's own training rows - are recomputed against the holdout fold. Carrying those
forward would fit a model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 期間，以及允許模型看到的資料

保留樣本期間不是在此決定的。其起訖值為 `evaluation.holdout_start` 與 `evaluation.holdout_end`，取自案例研究自身的 `setup.yaml`，並透過折推導與回測切片共同使用的 `canonical_window` 讀取，因此三者不可能不一致。

訓練區間涵蓋該期間開始前所有可用資料，其上界受標籤緩衝區限制。緩衝區可防止最後一筆訓練標籤的結果落在保留樣本內：本案例研究以做出決策的交易時段為每列標上日期，而標籤會從該時點向前看；因此，日期落在 2024-01-01 前一個預測期間內的資料列，其結果會在保留樣本期間實現。緩衝區因此需要涵蓋一個預測期間的交易時段：`fwd_ret_5d` 為 5 個，`fwd_ret_21d` 為 21 個。推導時採用所有聲明標籤中最長的期間封住邊界，而非只看目前擬合的標籤，確保任何標籤都不會跨過自身邊界。零間隔會造成資料洩漏，並非保守選擇，因此推導流程拒絕將其設為預設值。

設定的其他部分都原樣帶入；無法直接帶入的欄位，包括資格清單，以及此模型家族需根據各折自身訓練資料列決定的參數，則會針對保留樣本折重新計算。若照搬這些欄位，便是在擬合以驗證折為鍵值的模型，卻稱之為重新訓練。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains  {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends
# before the holdout opens. Printing both is what lets a reader check the gap rather than
# take it on the derivation's word.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")

Holdout fold 5
  trains  2011-01-04 -> 2023-11-29
  predicts 2024-01-01 -> 2025-12-31
  label buffer: 21D
Validation folds: 5, latest evaluation end 2023-11-29 00:00:00
Holdout training ends 2023-11-29, holdout opens 2024-01-01


## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the spec above. Its name
comes from the locked holdout path this case study no longer uses; it takes a training
specification and a checkpoint, not a lock, and it is used here because it is the one
call that refuses a request that is not exactly the spec it was handed - the training
identity, the checkpoint schedule, the feature lineage and the runtime parameters are
all checked before anything is fitted.

The training identity below is new. It has to be: it covers the CV interval, and the
holdout fold is not one of the validation folds. A run that came back with the
validation training hash would mean the refit did not happen.

**The window carries one configuration at a time.** The holdout is re-runnable, and that
is not the same as free: every configuration evaluated on it is another look at a period
the case study reports as unseen, and two evaluated quietly would make that report false.

So the check below is on the selected configuration rather than on the notebook, and it has
exactly two outcomes. With the selected configuration unchanged this is an idempotent replay: the
derivation is deterministic and the training identity covers it, so the same identity comes back
and the fit is served from the registry. With the selected configuration changed it refuses,
names both configurations, and stops.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 3. 擬合並登錄預測

`reconstruct_locked_model_request` 會依上方規格建立請求。其名稱來自本案例研究已不再使用的鎖定保留樣本路徑；它接受的是訓練規格與檢查點，而非鎖定值。此處使用它，是因為只有這個呼叫會拒絕與所給規格不完全一致的請求：訓練識別資訊、檢查點排程、特徵資料沿革與執行參數，都會在擬合前受到檢查。

下方的訓練識別資訊是新的，也必須是新的：它涵蓋交叉驗證區間，而保留樣本折不屬於任何驗證折。若執行結果仍帶有驗證期的訓練雜湊值，就表示沒有重新擬合。

**此期間一次只容許一項設定。** 保留樣本可以重新執行，但這不等於可以任意使用：每在其上評估一項設定，就多看了一次本案例研究聲稱尚未見過的期間；若悄悄評估兩項，這項聲明就不成立。

因此，下方的檢查針對的是選定設定，而非 Notebook，且只有兩種結果。若選定設定不變，這是冪等的重播：推導過程是確定的，訓練識別資訊也涵蓋該過程，因此會得到相同識別資訊，擬合結果直接由登錄資料庫提供。若選定設定改變，檢查會拒絕執行、列出兩項設定並停止。
<!-- ML4T_ZH_END c0010-p000 -->

It refuses rather than offering a replacement switch, and the reason is that a replacement
would not be one. Deleting the earlier generation's rows does not undo having observed its
result: the selection that produced the new configuration may have been informed by the old
holdout number, and no deletion reaches that. A switch here would let the case study take a
second look at the window while leaving a registry that shows only one, which is the
specific thing that would make the out-of-sample claim false rather than merely weak.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
系統會直接拒絕，而不提供替換開關，因為所謂替換並不能消除先前的觀察。刪除前一版本的資料列，不會抹去曾經看過其結果的事實：產生新設定的選擇，可能已受舊保留樣本數字影響，而刪除無法消除這種影響。若在此提供開關，本案例研究就能再次查看該期間，登錄資料庫卻只顯示一次；這會使樣本外主張失實，而不只是證據較弱。
<!-- ML4T_ZH_END c0010-p001 -->


In [7]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
superseded = [
    row
    for row in _registered_holdout_generations(CASE_DIR)
    if row["refitted"] and (row["training_hash"], row["checkpoint"]) != this_generation
]
if superseded:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the "
        "same window, which would be a second configuration measured on a period this case "
        "study reports as unseen. This notebook has no way past that: deleting the earlier "
        "generation would not undo having observed it, and the selection bias it introduces "
        "is not removed by removing the rows. Either leave the selection where it was, or "
        "retire the earlier evaluation through the registry's own lifecycle, which records "
        "that a second look was taken."
    )

In [8]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:  {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

Holdout training run:  8a956b00657c
Holdout prediction set: dab08ed3e7e9


What the prediction set covers, read back from the registry rather than from the
request. The two agree only if the fit published what it declared, and the count is the
one number a reader can check the window against: the holdout spans two years of
sessions, the strategy decides weekly, and the number of rows is those decision dates
times the products eligible on each.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
這組預測涵蓋的範圍，應從登錄資料庫讀回，而非只看請求。兩者只有在擬合程序確實發布了聲明內容時才會一致；資料列數是讀者唯一能用來核對期間的數字：保留樣本涵蓋兩年的交易時段，策略每週決策，而資料列數等於決策日期數乘以每個日期符合資格的商品數。
<!-- ML4T_ZH_END c0013-p000 -->


In [9]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
# `13_backtest` records that reader-facing rows use `product` and the shared boundary
# converts to the engine's `symbol` key, so a prediction set can arrive carrying either.
# Named from the frame rather than assumed, so this line cannot quietly report nothing.
_ENTITY_COL = next(c for c in ("product", "symbol") if c in predictions.columns)
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(f"rows={predictions.height:,}  dates={predictions['timestamp'].n_unique()}")
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions[_ENTITY_COL].n_unique():,} {_ENTITY_COL}s"
)

split=holdout  checkpoint=epoch=768
rows=14,760  dates=496
  2024-01-02 00:00:00+00:00 -> 2025-12-01 00:00:00+00:00, 30 symbols


The registry now holds more than one holdout prediction set for this case study, and
only one of them was fitted on data that ends before the window. The other is the
defective generation this notebook replaces: it carries the validation training identity,
which is how it was found. Both are listed rather than one silently preferred, because
the registry is immutable and a reader looking at it later will see both.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
登錄資料庫現在存有本案例研究不只一組保留樣本預測，而其中只有一組是在該期間開始前就結束的資料上擬合。另一組是本 Notebook 要取代的有瑕疵版本：它帶有驗證期的訓練識別資訊，問題因而被發現。此處會列出兩者，而非悄悄偏好其中一組，因為登錄資料庫不可變，日後查看的讀者也會看到兩組。
<!-- ML4T_ZH_END c0015-p000 -->


In [10]:
for row in _registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

  dab08ed3e7e9  training=8a956b00657c  sdf  refitted for the holdout


## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends before the window
opens. That is a precondition for an out-of-sample claim, not the claim itself. Nothing
here says whether the predictions are any good - they have not been scored, sized or
traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached
this notebook through a selection made on the validation folds, and this window is being
used once per configuration that gets here. What it does remove is the specific
circularity of scoring a validation-fitted model on the period meant to judge it.

The holdout is re-runnable. If a later pass finds the selection was wrong, the answer is
to delete this generation and produce another, not to treat the first as spent.

**Next:** [`18_holdout_backtest`](18_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 確立了什麼，以及沒有確立什麼

它只確立一件事：在保留樣本期間上的一組預測，由本案例研究選定的設定產生，並在期間開始前即已結束的資料上擬合。這是樣本外主張的先決條件，不是主張本身。此處沒有說明預測品質；它們尚未經過評分、決定部位規模或交易。

嚴格來說，它也無法讓保留樣本成為全新的測試。這項設定是經由驗證折上的選擇進入本 Notebook，而每項走到這裡的設定都會使用此期間一次。它消除的是特定循環問題：將在驗證期擬合的模型拿到原本應用來評判該模型的期間評分。

保留樣本可以重新執行。若之後發現選擇錯誤，處理方式是刪除這個版本並產生另一個，而不是把第一次使用視為已耗盡。

**下一步：**[`18_holdout_backtest`](18_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
